# 06 多卷积核计算

上一节我们学习了多通道卷积：输入有几个通道，一个卷积核就要有几层。

这一节继续学习另一个非常重要的问题：如果一层卷积里不只有一个卷积核，而是有多个卷积核，输出会是什么样？

这一节仍然只讲概念和计算规则，不写代码。

## 1. 为什么需要多个卷积核

一个卷积核可以理解成一种特征检测器。

它在图片上滑动，看看不同位置有没有某种局部特征。

但是一张图片里的特征不可能只有一种。

比如识别手写数字时，模型可能需要观察：

- 横线。
- 竖线。
- 斜线。
- 拐角。
- 闭合区域。
- 不同笔画的组合。

一个卷积核通常只能偏向检测某一类特征。

所以我们会使用多个卷积核，让它们分别学习不同的局部特征。

## 2. 一个卷积核输出一张特征图

先回顾上一节的结论：

```text
一个卷积核 -> 一张特征图 -> 一个输出通道
```

注意这里说的是一个完整卷积核。

如果输入是 RGB 图，一个卷积核内部会有 3 层权重。

但这 3 层权重会一起参与计算，最后合成一张特征图。

所以不管输入有几个通道，只要是一个卷积核，最后就输出一张特征图。

## 3. 多个卷积核输出多张特征图

如果一层卷积里有多个卷积核，每个卷积核都会独立地在输入上滑动。

每个卷积核都会产生一张自己的特征图。

比如有 4 个卷积核：

```text
第 1 个卷积核 -> 第 1 张特征图
第 2 个卷积核 -> 第 2 张特征图
第 3 个卷积核 -> 第 3 张特征图
第 4 个卷积核 -> 第 4 张特征图
```

最后把这 4 张特征图叠在一起，就形成 4 个输出通道。

所以：

```text
有几个卷积核，就有几个输出通道。
```

## 4. 多卷积核不是重复卷积同一张特征图

这里有一个容易误会的地方。

多个卷积核不是第 1 个卷积核算完后，把结果交给第 2 个卷积核继续算。

在同一层卷积里，多个卷积核通常是并列工作的。

它们看到的是同一个输入。

区别是：每个卷积核有自己的参数，所以关注的特征可能不同。

可以这样理解：

```text
同一张输入图像
    -> 卷积核 1 看一种特征
    -> 卷积核 2 看另一种特征
    -> 卷积核 3 看第三种特征
    -> ...
```

最后这些特征图一起组成这一层的输出。

## 5. 输出通道数由谁决定

多卷积核计算里最重要的规则是：

```text
输出通道数 = 卷积核个数
```

例如：

```text
1 个卷积核  -> 1 个输出通道
6 个卷积核  -> 6 个输出通道
16 个卷积核 -> 16 个输出通道
32 个卷积核 -> 32 个输出通道
```

所以 CNN 中常说的输出通道数，本质上就是这一层用了多少个卷积核。

## 6. 输入通道数和输出通道数不要混淆

输入通道数和输出通道数是两个不同概念。

输入通道数由输入数据或上一层决定。

输出通道数由当前层卷积核个数决定。

可以这样记：

```text
输入有多少通道，决定每个卷积核有多深。
当前有多少卷积核，决定输出有多少通道。
```

比如输入是 RGB 图片，有 3 个输入通道。

如果这一层使用 16 个卷积核，那么输出通道数就是 16。

也就是说：

$$
3\times H\times W \rightarrow 16\times H_{out}\times W_{out}
$$

## 7. 一个具体例子：灰度图输入

假设输入是一张 MNIST 灰度图：

$$
1\times28\times28
$$

这里输入通道数是 1。

如果使用 6 个 5 x 5 的卷积核，不加 padding，stride = 1。

每个卷积核都会输出一张 24 x 24 的特征图。

因为有 6 个卷积核，所以一共会得到 6 张特征图。

输出形状就是：

$$
6\times24\times24
$$

这里的 6 表示模型提取出了 6 类特征。

## 8. 一个具体例子：RGB 图输入

假设输入是一张 RGB 图片：

$$
3\times32\times32
$$

输入通道数是 3。

如果使用 16 个 3 x 3 的卷积核，并且 padding = 1、stride = 1，高宽保持不变。

每个卷积核都会输出一张 32 x 32 的特征图。

因为有 16 个卷积核，所以输出是：

$$
16\times32\times32
$$

注意：输入通道是 3，输出通道是 16。

这说明输出通道数不一定等于输入通道数。

## 9. 加上 batch 后怎么表示

前面的形状都是单张图片。

训练时通常是一批图片一起输入。

如果 batch size 是 64，输入 RGB 图片形状是：

$$
64\times3\times32\times32
$$

经过 16 个卷积核后，如果高宽保持不变，输出就是：

$$
64\times16\times32\times32
$$

这里 batch size 没变。

变化的是通道数：从 3 变成 16。

高和宽是否变化，则由卷积核大小、padding 和 stride 决定。

## 10. 多卷积核的参数量

多卷积核会带来更多参数。

假设输入通道数是 $C_{in}$，卷积核大小是 K x K。

一个卷积核的权重数量是：

$$
C_{in}\times K\times K
$$

如果有 $C_{out}$ 个卷积核，总权重数量就是：

$$
C_{out}\times C_{in}\times K\times K
$$

如果每个卷积核还有一个偏置，总参数量就是：

$$
C_{out}\times C_{in}\times K\times K + C_{out}
$$

这里的 $C_{out}$ 既表示卷积核个数，也表示输出通道数。

## 11. 参数量例子

假设输入是 RGB 图片，所以输入通道数是 3。

卷积核大小是 3 x 3。

这一层使用 16 个卷积核。

一个卷积核的权重数量是：

$$
3\times3\times3=27
$$

16 个卷积核的权重数量是：

$$
16\times27=432
$$

如果每个卷积核有一个偏置，偏置数量是 16。

所以总参数量是：

$$
432+16=448
$$

这说明卷积层参数量和卷积核个数有关，但仍然通过权值共享避免了像全连接层那样参数爆炸。

## 12. 多卷积核和特征种类

为什么卷积核越多，输出通道越多？

可以把每个输出通道理解成一种特征响应。

比如：

```text
第 1 个输出通道：可能更关注横向边缘
第 2 个输出通道：可能更关注竖向边缘
第 3 个输出通道：可能更关注斜向边缘
第 4 个输出通道：可能更关注局部拐角
...
```

这些具体关注什么，不是我们手工指定的，而是模型在训练过程中学出来的。

所以输出通道数越多，模型在这一层可以保留的特征种类通常越多。

但卷积核也不是越多越好。

卷积核越多，参数量和计算量也会增加。

## 13. 多卷积核在 CNN 里的常见变化

CNN 中常见的通道变化是逐渐增多。

例如：

```text
输入图片：1 个通道
第一层卷积后：16 个通道
第二层卷积后：32 个通道
第三层卷积后：64 个通道
```

这可以理解成：越往后，模型提取的特征种类越丰富。

前面的层可能看简单边缘。

后面的层可能看更复杂的局部形状。

所以通道数增加，通常表示模型在更高层次上保留更多特征。

## 14. 容易混淆的地方

第一，多个卷积核不是多个输入图片。

它们是在同一个输入上并列提取不同特征。

第二，输出通道数不是由输入图片颜色决定的。

RGB 输入有 3 个颜色通道，但卷积后可以输出 16、32、64 个特征通道。

第三，一个卷积核虽然可能很深，但它只输出一张特征图。

卷积核的深度由输入通道数决定，卷积核的个数才决定输出通道数。

第四，多个卷积核参数互不相同。

如果所有卷积核参数都一样，那它们学到的特征就会重复，模型也就没有必要保留这么多输出通道。

## 15. 本节小结

这一节先记住这些规则：

1. 一个卷积核产生一张特征图。
2. 多个卷积核产生多张特征图。
3. 多张特征图叠起来，形成多个输出通道。
4. 输出通道数等于卷积核个数。
5. 输入通道数决定每个卷积核有多深。
6. 输出通道数决定这一层要准备多少个卷积核。
7. 多个卷积核可以学习不同种类的特征。
8. 卷积核越多，表达能力可能更强，但参数量和计算量也会增加。

## 自检问题

1. 为什么一层卷积里通常需要多个卷积核？
2. 一个卷积核会产生几张特征图？
3. 16 个卷积核会产生几个输出通道？
4. 输入通道数由什么决定？
5. 输出通道数由什么决定？
6. 输入是 3 x 32 x 32，使用 16 个卷积核，高宽保持不变，输出形状是什么？
7. batch size 是 64 时，上一个问题的输出形状是什么？
8. 为什么输出的 16 个通道不是 16 个颜色通道？
9. 多个卷积核是前后串行计算，还是在同一层里并列提取特征？
10. 卷积核数量增加会带来什么好处和代价？